In [ ]:
import cv2
import numpy as np
import os
# Класс для предобработки
class VideoPreprocessor:
    def __init__(self, video_path: str):
        self.video_path = video_path
        self.cap = cv2.VideoCapture(video_path)
        
        if not self.cap.isOpened():
            raise ValueError(f"Не удалось открыть видео: {video_path}")
        
        # Вытаскиваем метаданные
        self.fps = self.cap.get(cv2.CAP_PROP_FPS) # fps
        self.total_frames = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT)) # Количество кадров
        self.duration_ms = int((self.total_frames / self.fps) * 1000) # Продолжитеьлность
        
        # Параметры
        self.window_size_ms = 1000  # 1 секунда
        self.sharpness_threshold = 100.0 # Порог резкости
        self.rotation_code = cv2.ROTATE_90_COUNTERCLOCKWISE # Переворачиваем на 90  градусов
    
    def calculate_sharpness(self, frame):
        # Для каждого кадра считаем резкость по метрике лапласиана
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        laplacian = cv2.Laplacian(gray, cv2.CV_64F)
        return laplacian.var() # Возращаем дисперсию
    
    def enhance_frame(self, frame):
        # Считаем CLAHE лдя каждого кадра, чтобы повыстить контрастность
        # lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB) # lab для того, чтобы не трогать цвета, а только яркость
        # clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        # lab[:,:,0] = clahe.apply(lab[:,:,0])
        # frame = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
        
        # # Осветляем тени
        # gamma = 0.9
        # inv_gamma = 1.0 / gamma
        # table = np.array([((i / 255.0) ** (1.0 / inv_gamma) * 255)
        #                   for i in np.arange(0, 256)]).astype("uint8")
        # frame = cv2.LUT(frame, table)
        
        # # Фильтр для удалпения шума
        # frame = cv2.bilateralFilter(frame, d=5, sigmaColor=50, sigmaSpace=50)

    # def enhance_frame_adaptive(frame):
        lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB) # lab для того, чтобы не трогать цвета, а только яркость

        L = lab[:,:,0].astype(float)
        
        # Оцениваем долю засветов (пиксели > 240)
        glare_ratio = np.sum(L > 240) / L.size
        
        # Адаптивные параметры
        if glare_ratio > 0.15:  # >15% кадра в бликах
            clip_limit = 1.5    # мягче CLAHE
            gamma = 1.1         # чуть затемняем, чтобы вытянуть детали из пересвета
        else:
            clip_limit = 2.0
            gamma = 0.9
            
        # Подавление бликов (простое сжатие ярких пикселей)
        L = np.where(L > 230, L * 0.85, L)
        L = np.clip(L, 0, 255).astype(np.uint8)
        
        # CLAHE
        clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(8,8))
        lab[:,:,0] = clahe.apply(L)
        
        # Осветляем тени
        inv_gamma = 1.0 / gamma
        table = np.array([((i / 255.0) ** (1.0 / inv_gamma) * 255)
                        for i in np.arange(0, 256)]).astype("uint8")
        lab[:,:,0] = cv2.LUT(lab[:,:,0], table)
        
        # Фильтр для удалпения шума
        enhanced = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
        return cv2.bilateralFilter(enhanced, d=5, sigmaColor=50, sigmaSpace=50)


        

    
    # Пайплайн предобработки 
    def process(self):

            # Пайплайн такой: читаем видео, поварачиваем, разиваем на окна, в каждом окне выбираем лучший кадр, улучшаем его, возращаем список (timestamp, frame, sharpness)

            best_frames = [] # Лучшие кадры
            buffer = []  # Кадры в текущем окне
            last_flush_ts = 0
            
            frame_idx = 0
            
            while self.cap.isOpened():
                ret, frame = self.cap.read()
                if not ret:
                    break
                
                # Поворот
                frame = cv2.rotate(frame, self.rotation_code)
                
                # Timestamp
                ts_ms = int(self.cap.get(cv2.CAP_PROP_POS_MSEC))
                
                # Sharpness
                sharpness = self.calculate_sharpness(frame)
                
                # Фильтруем размытые
                if sharpness > self.sharpness_threshold:
                    buffer.append((ts_ms, sharpness, frame.copy()))
                
                # Выбираем лучшее в каждом окне
                if ts_ms - last_flush_ts >= self.window_size_ms:
                    if buffer:
                        # Лучший по sharpness
                        best_ts, best_score, best_img = max(buffer, key=lambda x: x[1])
                        
                        # Улучшаем
                        enhanced = self.enhance_frame(best_img)
                        best_frames.append((best_ts, enhanced, best_score))
                    
                    # Сброс
                    buffer = []
                    last_flush_ts = ts_ms
                
                frame_idx += 1
            
            # Последний кадр
            self.cap.release()
            
            return best_frames
    
    def save_frames(self, frames: list, output_dir: str):
        """Сохраняем кадры для отладки"""
        os.makedirs(output_dir, exist_ok=True)
        
        for i, (ts, frame, score) in enumerate(frames):
            filename = f"frame_{ts}_score_{score:.1f}.jpg"
            filepath = os.path.join(output_dir, filename)
            cv2.imwrite(filepath, frame)
        
        print(f"Сохранено {len(frames)} кадров в {output_dir}")


# Использование
if __name__ == "__main__":
    preprocessor = VideoPreprocessor(r"25_2-10.mp4")
    frames = preprocessor.process()
    preprocessor.save_frames(frames, "extracted_frames")
    
    print(f"\nСтатистика:")
    print(f"  Всего кадров: {len(frames)}")
    print(f"  Средняя чёткость: {np.mean([s for _,_,s in frames]):.1f}")
    print(f"  Диапазон: {min([s for _,_,s in frames]):.1f} – {max([s for _,_,s in frames]):.1f}")

In [ ]:
import cv2
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from tqdm import tqdm
import os

# Настройки
VIDEO_PATH = "data/43_15.mp4"
MODEL_PATH = r"D:\hak\Lenta Tech\Lenta_Tech_detector\runs\detect\models\price_tag_v1\weights\best.pt"
CROPS_DIR = "crops"
os.makedirs(CROPS_DIR, exist_ok=True)
CONF_THRESH = 0.05

print(" Загрузка модели...")
model = YOLO(MODEL_PATH, verbose=False)

cap = cv2.VideoCapture(VIDEO_PATH)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
saved = 0

print(f" Обработка видео ({total_frames} кадров)...")
for idx in tqdm(range(total_frames), desc="Инференс"):
    ret, frame = cap.read()
    if not ret: break
    
    # Поворот 90° против часовой (ТЗ)
    frame = cv2.rotate(frame, cv2.ROTATE_90_COUNTERCLOCKWISE)
    ts_ms = int(cap.get(cv2.CAP_PROP_POS_MSEC))
    
    # Детекция
    results = model(frame, conf=CONF_THRESH, verbose=False)
    
    for i, box in enumerate(results[0].boxes):
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
        
        # Базовый фильтр геометрии (убираем явный мусор)
        bw, bh = x2-x1, y2-y1
        if bw*bh < 2000: continue
        if not (1.0 < bw/max(bh,1) < 5.0): continue
        
        # Вырезаем и сохраняем кроп
        crop = frame[y1:y2, x1:x2]
        if crop.size == 0: continue
        
        fname = f"frame_{idx:05d}_ts{ts_ms}_bbox_{x1}_{y1}_{x2}_{y2}.jpg"
        cv2.imwrite(os.path.join(CROPS_DIR, fname), crop)
        saved += 1

cap.release()
print(f" Сохранено {saved} кропов в папке {CROPS_DIR}/")